# CDR-MLC versus a single global Random Forest

This notebook compares leakage-safe hard-routed CDR-MLC with one Random Forest trained on all training samples.

Fairness controls:

- both methods use exactly the same 32 classification features;
- both use 80 trees, balanced class weights, and random seed 42;
- the global RF does not use the three congestion-routing signals (SynAck, AckDat, TcpRtt);
- IdleTime, the target, and target-derived metadata are excluded;
- every scenario uses a separate training and test file.

Set DATA_DIR to the folder containing the five scale_0.001 CSV files.


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import LabelEncoder

DATA_DIR = Path("DATASETS/CDR-MLC/scale_0.001")
MAIN = Path("CDR-MLC.ipynb")
if not MAIN.exists():
    MAIN = Path("CDR_MLC") / "CDR-MLC.ipynb"
namespace = {}
with MAIN.open(encoding="utf-8") as handle:
    notebook = json.load(handle)
exec(compile("".join(notebook["cells"][0]["source"]), str(MAIN), "exec"), namespace)
run_pipeline_from_two_files = namespace["run_pipeline_from_two_files"]

try:
    display
except NameError:
    display = print


def _is_target_metadata(column):
    name = column.strip().lower().replace("-", "_").replace(" ", "_")
    return (
        "label" in name or name in {"class", "target", "y"}
        or name.startswith(("class_", "target_", "index_in_", "unnamed:"))
    )


def compare_cdr_mlc_with_global_rf(scenario, train_file, test_file):
    cdr = run_pipeline_from_two_files(
        str(train_file), str(test_file), n_clusters=3, window_size=3,
        clustering_stats=["mean", "median", "std", "min", "max"],
    )

    train = pd.read_csv(train_file)
    test = pd.read_csv(test_file)
    for frame in (train, test):
        if "IdleTime" in frame.columns:
            frame.drop(columns=["IdleTime"], inplace=True)

    target = "label"
    fixed_routing_features = ["SynAck", "AckDat", "TcpRtt"]
    features = [
        column for column in train.select_dtypes(include=[np.number]).columns
        if column not in fixed_routing_features
        and not _is_target_metadata(column)
    ]
    assert features == cdr["classification_features"]
    assert len(features) == 32

    encoder = LabelEncoder()
    y_train = encoder.fit_transform(train[target])
    y_test = encoder.transform(test[target])
    global_rf = RandomForestClassifier(
        n_estimators=80, random_state=42,
        class_weight="balanced", n_jobs=-1,
    )
    global_rf.fit(train[features], y_train)
    predictions = global_rf.predict(test[features])

    rf_metrics = {
        "accuracy": accuracy_score(y_test, predictions),
        "precision_weighted": precision_score(
            y_test, predictions, average="weighted", zero_division=0),
        "recall_weighted": recall_score(
            y_test, predictions, average="weighted"),
        "f1_weighted": f1_score(
            y_test, predictions, average="weighted"),
        "f1_macro": f1_score(y_test, predictions, average="macro"),
    }
    cdr_metrics = cdr["test_results"]
    comparison = pd.DataFrame([
        {"method": "global_rf", **rf_metrics},
        {"method": "cdr_mlc_hard", **{
            key: cdr_metrics[key] for key in rf_metrics}},
    ])
    for metric in rf_metrics:
        comparison[f"{metric}_gain_pp_vs_rf"] = (
            100 * (comparison[metric] - rf_metrics[metric]))
    print(f"\n{scenario}")
    display(comparison.round(4))
    return {"cdr": cdr, "global_rf": global_rf,
            "features": features, "comparison": comparison}


In [ ]:
# scenario_1: run independently
scenario_1_comparison = compare_cdr_mlc_with_global_rf(
    "scenario_1", DATA_DIR / "level_1.csv", DATA_DIR / "level_2.csv",
)


In [ ]:
# scenario_2: run independently
scenario_2_comparison = compare_cdr_mlc_with_global_rf(
    "scenario_2", DATA_DIR / "level_1.csv", DATA_DIR / "level_3.csv",
)


In [ ]:
# scenario_3: run independently
scenario_3_comparison = compare_cdr_mlc_with_global_rf(
    "scenario_3", DATA_DIR / "level_2.csv", DATA_DIR / "level_3.csv",
)


In [ ]:
# scenario_4: run independently
scenario_4_comparison = compare_cdr_mlc_with_global_rf(
    "scenario_4", DATA_DIR / "CDR-MLC-Shuffle.csv", DATA_DIR / "CDR-MLC-Shuffle(1).csv",
)


In [ ]:
# scenario_5: run independently
scenario_5_comparison = compare_cdr_mlc_with_global_rf(
    "scenario_5", DATA_DIR / "CDR-MLC-Shuffle(1).csv", DATA_DIR / "CDR-MLC-Shuffle.csv",
)
